# PART4: GenAI: turn call-centre notes into structured reasons, then recommend

In [ ]:
#Task 4.1 — One validated call
#installing genai
#initialize client using getpass
!pip install -q google-genai pydantic

from getpass import getpass
import json
import pandas as pd
from google import genai
from google.genai import types

API_KEY = getpass("Paste your key: ").strip()
client = genai.Client(api_key=API_KEY)
MODEL_NAME = "gemini-3.8-flash"

Paste your key: ··········


## Task 4.1 — Gemini Client Setup & Single Validated Call
- Install `google-genai` and `pydantic`, securely load the API key via `getpass.getpass()` (keeping credentials out of the notebook), and initialize the Gemini client.
- Define a Pydantic output schema for structured note extraction and run a single validated API call on a sample call-centre note[span_0](start_span)[span_0](end_span).
-

In [ ]:
## thinking_config is omitted because gemini-flash-lite-latest does not support thinking_level
import json
import time
import pandas as pd
from google.genai import types
from google.genai.errors import ServerError

MODEL_NAME = "gemini-flash-lite-latest"

SYSTEM_TEXT = (
    "You are a healthcare clinic operations analyst. "
    "Classify patient follow-up call notes accurately in a concise, objective tone."
)

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

def analyse_note(note_text):
    prompt = f"""
Context: A clinic call-centre logged a short note after calling a patient who missed an appointment.
Role: Operational analyst categorizing no-show reasons for clinic management.
Action: Read the note below, classify it into one allowed root_cause, provide a confidence score between 0 and 1, write a summary of at most 25 words, and suggest one practical recommended_action.
Allowed root_cause values: forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, work_or_schedule, other.
Format: Return a JSON object with keys root_cause, confidence, summary, and recommended_action.
Tone: Factual, concise, and professional.

Note: "{note_text}"
"""
    for attempt in range(4):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_TEXT,
                    temperature=0,
                    max_output_tokens=1000,
                    response_mime_type="application/json",
                    response_schema=SCHEMA,
                ),
            )
            return json.loads(response.text)
        except ServerError:
            if attempt == 3:
                raise
            time.sleep(2 ** (attempt + 1))

# Run on note N001 and print the parsed dictionary
notes_df = pd.read_csv("followup_notes.csv")
n001_text = notes_df.loc[notes_df["note_id"] == "N001", "note_text"].iloc[0]

result_n001 = analyse_note(n001_text)
print("Note N001:", n001_text)
print("Parsed result:", json.dumps(result_n001, indent=2))

Note N001: Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.
Parsed result: {
  "root_cause": "forgot_or_no_reminder",
  "confidence": 0.98,
  "summary": "Patient missed appointment due to lack of reminder and forgetting the date, and requested rescheduling.",
  "recommended_action": "Verify patient contact preferences and update notification settings for future appointments."
}


## Task 4.1 (Cont.) — Structured JSON Extraction Function (`analyse_note`)
- Define `SCHEMA` (`root_cause`, `confidence`, `summary`, `recommended_action`) and implement `analyse_note(note_text)` using `gemini-flash-lite-latest` with `response_mime_type="application/json"`, `temperature=0`, and exponential backoff retries on `ServerError`.
- Test the pipeline on note `N001` from `followup_notes.csv` and verify the parsed JSON output (`root_cause: "forgot_or_no_reminder"`).
-

In [ ]:
#Task 4.2 — Validate and process the 10 notes
import json
import time
import pandas as pd
from pydantic import BaseModel, Field, ValidationError

ALLOWED_CAUSES = {
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
}


# (a) Define Pydantic model FollowupInsight
class FollowupInsight(BaseModel):
  root_cause: str = Field(min_length=3)
  confidence: float = Field(ge=0, le=1)
  summary: str = Field(min_length=5)
  recommended_action: str = Field(min_length=5)


# (b) & (c) Read followup_notes.csv and loop over first 10 rows with validation and 13s pause
notes_df = pd.read_csv("followup_notes.csv")
first_10 = notes_df.head(10)

records = []

for _, row in first_10.iterrows():
  note_id = row["note_id"]
  appointment_id = row["appointment_id"]
  note_text = row["note_text"]

  # Input validation: check pd.isna() first, then strip()
  if pd.isna(note_text) or str(note_text).strip() == "":
    records.append({
        "note_id": note_id,
        "appointment_id": appointment_id,
        "root_cause": "other",
        "confidence": 0.0,
        "summary": "No note recorded",
        "recommended_action": "Re-call patient",
        "validation_status": "skipped_blank",
        "raw_response": "",
    })
    continue

  # Call analyse_note() from Task 4.1
  raw_reply = ""
  data = {}
  try:
    data = analyse_note(str(note_text).strip())
    raw_reply = json.dumps(data)

    # Validate with Pydantic + allowed root_cause list
    insight = FollowupInsight(**data)
    if insight.root_cause not in ALLOWED_CAUSES:
      raise ValueError(f"Invalid root_cause: {insight.root_cause}")

    status = "valid"
  except (ValidationError, ValueError, TypeError, Exception):
    status = "invalid"

  # Keep reply exactly as returned without repairing missing/invalid fields
  records.append({
      "note_id": note_id,
      "appointment_id": appointment_id,
      "root_cause": data.get("root_cause", "") if isinstance(data, dict) else "",
      "confidence": data.get("confidence", "") if isinstance(data, dict) else "",
      "summary": data.get("summary", "") if isinstance(data, dict) else "",
      "recommended_action": (
          data.get("recommended_action", "") if isinstance(data, dict) else ""
      ),
      "validation_status": status,
      "raw_response": raw_reply,
  })

  # Pause 13 seconds between API calls to respect free-tier rate limit
  time.sleep(13)

# (d) Save notes_analysed.csv
analysed_df = pd.DataFrame(
    records,
    columns=[
        "note_id",
        "appointment_id",
        "root_cause",
        "confidence",
        "summary",
        "recommended_action",
        "validation_status",
        "raw_response",
    ],
)
analysed_df.to_csv("notes_analysed.csv", index=False)

print("Saved notes_analysed.csv with shape:", analysed_df.shape)
print(
    analysed_df[
        ["note_id", "appointment_id", "root_cause", "confidence", "validation_status"]
    ].to_string(index=False)
)

Saved notes_analysed.csv with shape: (10, 8)
note_id appointment_id            root_cause  confidence validation_status
   N001          A0872 forgot_or_no_reminder        0.98             valid
   N002          A0897 distance_or_transport        0.98             valid
   N003          A0526           cost_or_fee        0.98             valid
   N004          A0926                 other        0.00     skipped_blank
   N005          A0534  illness_or_emergency        0.98             valid
   N006          A0507      work_or_schedule        0.98             valid
   N007          A0333 forgot_or_no_reminder        0.95             valid
   N008          A0222 distance_or_transport        0.98             valid
   N009          A0977                 other        0.95             valid
   N010          A0596      work_or_schedule        0.95             valid


## Task 4.2 — Pydantic Validation & Batch Processing (`notes_analysed.csv`)
- Define the `FollowupInsight` Pydantic schema and `ALLOWED_CAUSES` set, handle blank or missing notes as `"skipped_blank"`, and run `analyse_note()` across the first 10 rows of `followup_notes.csv` with a 13-second rate-limit pause between API calls.
- Validate every response against `FollowupInsight` (`valid` vs. `invalid`), preserve raw model replies without manual repair, and save the structured output to `notes_analysed.csv`.
-

In [ ]:
#Task 4.3 — Break it on purpose

import json
import time
import pandas as pd
from google.genai import types
from google.genai.errors import ServerError
from pydantic import ValidationError

# --- (a) Feed two deliberately bad replies to Pydantic and catch ValidationError ---
bad_confidence_reply = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot the appointment time.",
    "recommended_action": "Send automated SMS reminder.",
}

missing_action_reply = {
    "root_cause": "cost_or_fee",
    "confidence": 0.92,
    "summary": "Patient could not afford consultation fee.",
}

print("=== Task 4.3 (a): Pydantic ValidationError Tests ===")
for label, bad_data in [
    ("Bad confidence (1.7)", bad_confidence_reply),
    ("Missing recommended_action", missing_action_reply),
]:
  try:
    FollowupInsight(**bad_data)
  except ValidationError as err:
    print(f"\n[{label}] Caught ValidationError:")
    print(err)

=== Task 4.3 (a): Pydantic ValidationError Tests ===

[Bad confidence (1.7)] Caught ValidationError:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

[Missing recommended_action] Caught ValidationError:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'cost_or_f...ford consultation fee.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


## Task 4.3 (a) — Deliberate Schema Failure & `ValidationError` Testing
- Pass two intentionally malformed payloads into `FollowupInsight`—an out-of-range confidence score (`confidence: 1.7`) and a missing required field (`recommended_action`)—to confirm Pydantic catches and reports `ValidationError` exceptions cleanly.
-

In [ ]:
# --- (b) Compare V1 vs V2 prompts on N007 (Hinglish) and N009 (ambiguous) ---
# Uses the exact same SCHEMA, JSON mode, and temperature=0 for both prompts
# Also adds an explicit instruction in V2 to answer 'other' rather than guess when ambiguous.


def call_with_prompt(prompt_text):
  for attempt in range(4):
    try:
      resp = client.models.generate_content(
          model=MODEL_NAME,
          contents=prompt_text,
          config=types.GenerateContentConfig(
              system_instruction=SYSTEM_TEXT,
              temperature=0,
              max_output_tokens=1000,
              response_mime_type="application/json",
              response_schema=SCHEMA,
          ),
      )
      return json.loads(resp.text)
    except ServerError:
      if attempt == 3:
        raise
      time.sleep(2 ** (attempt + 1))


notes_df = pd.read_csv("followup_notes.csv")
comparison_rows = []
api_calls_4_3 = 0

for target_id in ["N007", "N009"]:
  note_txt = str(
      notes_df.loc[notes_df["note_id"] == target_id, "note_text"].iloc[0]
  ).strip()

  # V1: Minimal prompt ("Classify this note")
  v1_prompt = f'Classify this note:\n\nNote: "{note_txt}"'
  v1_res = call_with_prompt(v1_prompt)
  api_calls_4_3 += 1
  time.sleep(13)

  # V2: Full CRAFT prompt with allowed values + instruction to answer 'other' rather than guess
  v2_prompt = f"""
Context: A clinic call-centre logged a short note after calling a patient who missed an appointment.
Role: Operational analyst categorizing no-show reasons for clinic management.
Action: Read the note below, classify it into one allowed root_cause, provide a confidence score between 0 and 1, write a summary of at most 25 words, and suggest one practical recommended_action. If the note is unclear or ambiguous, answer "other" rather than guessing.
Allowed root_cause values: forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, work_or_schedule, other.
Format: Return a JSON object with keys root_cause, confidence, summary, and recommended_action.
Tone: Factual, concise, and professional.

Note: "{note_txt}"
"""
  v2_res = call_with_prompt(v2_prompt)
  api_calls_4_3 += 1
  time.sleep(13)

  for ver, res in [("V1 (Minimal)", v1_res), ("V2 (CRAFT)", v2_res)]:
    rc = res.get("root_cause", "")
    comparison_rows.append({
        "note_id": target_id,
        "prompt_version": ver,
        "root_cause": rc,
        "in_allowed_list": rc in ALLOWED_CAUSES,
        "confidence": res.get("confidence", ""),
        "summary": res.get("summary", ""),
    })

comp_df = pd.DataFrame(comparison_rows)
print("\n=== Task 4.3 (b): Prompt Comparison (V1 vs V2) ===")
print(comp_df.to_string(index=False))

print("\nDeployment Recommendation:")
print(
    "Deploy V2 (CRAFT prompt). V1 lacks explicit category boundaries and fallback rules, "
    "risking out-of-schema labels or forced guesses on Hinglish (N007) and ambiguous notes (N009). "
    "V2 enforces the 6 allowed root_cause categories and safely defaults ambiguous cases to 'other'."
)


=== Task 4.3 (b): Prompt Comparison (V1 vs V2) ===
note_id prompt_version                                                                                    root_cause  in_allowed_list  confidence                                                                                                              summary
   N007   V1 (Minimal) Patient forgot the appointment due to not receiving the SMS reminder and needs to reschedule.            False        0.98                      Patient missed the appointment because no SMS reminder was received and requests a new booking.
   N007     V2 (CRAFT)                                                                         forgot_or_no_reminder             True        0.95 Patient missed the appointment because they forgot and did not receive an SMS reminder. They requested rescheduling.
   N009   V1 (Minimal)                                                                                   unreachable            False        1.00              

## Task 4.3 (b) — Prompt Comparison (`V1 Minimal` vs. `V2 CRAFT`)
- Evaluate a minimal prompt (`V1`) against a structured CRAFT prompt (`V2`) using the same JSON schema and `temperature=0` on Hinglish (`N007`) and ambiguous (`N009`) call notes.
- Compare schema compliance (`in_allowed_list`) and confidence scores side by side, confirming that `V2` enforces the 6 allowed `root_cause` categories and safely defaults ambiguous notes to `"other"` rather than guessing.
-

In [ ]:
# --- (c) Count and print total API calls across Tasks 4.1 - 4.3 ---
calls_4_1 = 1  # Single call for N001
calls_4_2 = int((analysed_df["validation_status"] != "skipped_blank").sum())  # 9 non-blank calls
total_api_calls = calls_4_1 + calls_4_2 + api_calls_4_3

print("\n=== Task 4.3 (c): Total API Call Count (Tasks 4.1 - 4.3) ===")
print(f"Task 4.1 API calls: {calls_4_1}")
print(f"Task 4.2 API calls: {calls_4_2} (N004 skipped as blank)")
print(f"Task 4.3 API calls: {api_calls_4_3} (2 notes x 2 prompt versions)")
print(f"Total API calls across Tasks 4.1 - 4.3: {total_api_calls}")


=== Task 4.3 (c): Total API Call Count (Tasks 4.1 - 4.3) ===
Task 4.1 API calls: 1
Task 4.2 API calls: 9 (N004 skipped as blank)
Task 4.3 API calls: 4 (2 notes x 2 prompt versions)
Total API calls across Tasks 4.1 - 4.3: 14


## Task 4.3 (c) — Total Gemini API Call Count (Tasks 4.1 – 4.3)
- Tally all Gemini API requests across Part 4: 1 call in Task 4.1 (`N001`), 9 non-blank calls in Task 4.2 (`N004` skipped as blank), and 4 calls in Task 4.3 (`2 notes × 2 prompt versions`), confirming a total of **14 API calls** within the free-tier limit.


In [ ]:
#Task 4.4 — Recommendation and GitHub
import pandas as pd

analysed = pd.read_csv("notes_analysed.csv", keep_default_na=False)
appts = pd.read_csv("appointments_clean.csv")

merged_df = pd.merge(analysed, appts, on="appointment_id", how="inner")
print("Merged shape:", merged_df.shape)
print(
    merged_df[
        ["note_id", "appointment_id", "root_cause", "reminder_sent", "status"]
    ].to_string(index=False)
)

ct = pd.crosstab(
    merged_df["root_cause"], merged_df["reminder_sent"], margins=True
)
print("\n=== Cross-Tabulation: root_cause vs reminder_sent ===")
print(ct)

Merged shape: (10, 23)
note_id appointment_id            root_cause reminder_sent  status
   N001          A0872 forgot_or_no_reminder            No No-Show
   N002          A0897 distance_or_transport            No No-Show
   N003          A0526           cost_or_fee           Yes No-Show
   N004          A0926                 other            No No-Show
   N005          A0534  illness_or_emergency           Yes No-Show
   N006          A0507      work_or_schedule           Yes No-Show
   N007          A0333 forgot_or_no_reminder            No No-Show
   N008          A0222 distance_or_transport           Yes No-Show
   N009          A0977                 other           Yes No-Show
   N010          A0596      work_or_schedule            No No-Show

=== Cross-Tabulation: root_cause vs reminder_sent ===
reminder_sent          No  Yes  All
root_cause                         
cost_or_fee             0    1    1
distance_or_transport   1    1    2
forgot_or_no_reminder   2    0    2
illne

## Task 4.4 — Cross-Tabulation & Operational Recommendations
- Inner-join `notes_analysed.csv` (`keep_default_na=False`) with `appointments_clean.csv` on `appointment_id` to link extracted LLM root causes with appointment metadata.
- Cross-tabulate `root_cause` against `reminder_sent` (`pd.crosstab(..., margins=True)`) to pinpoint reminder gaps and ground actionable clinic workflow recommendations[span_0](start_span)[span_0](end_span).


Takeaway (Task 4.4): Merging the 10 analysed notes with appointments_clean.csv shows that 5 of the 10 missed appointments had reminder_sent = No, including both (2 of 2) appointments classified as forgot_or_no_reminder (N001 and N007), directly linking operational reminder gaps to preventable no-shows.